# Bayesian Networks and Autoregressive Language Models

**Course:** CS F407 — Artificial Intelligence · **Lab:** Bayesian Networks
**Handout:** `BN_lab.pdf`

---

The progression this lab builds:

$$\text{Probability} \to \text{Bayesian Network} \to \text{Autoregressive Model} \to \text{Language Generation}$$

Everything here is plain Python — `dict`s, `Counter`s and `random` — with no ML
library and no pretrained model, exactly as the handout requires. That is the
point: the probabilistic structure has to be visible, and a neural network would
hide it.

Two things go beyond the literal instructions, because they turn the handout's
warnings into measurements:

1. **Part VII asks what a total of 0.87 would tell you.** Rather than only
   checking that the correct model normalises, the notebook builds a
   **deliberately broken** model with the classic off-by-one bug and shows its
   totals coming out wrong — so the diagnostic is demonstrated, not described.
2. **Part XIII asks for a first- vs. second-order comparison.** The notebook
   measures held-out perplexity alongside parameter counts and sentence
   diversity, which is what makes the data-sparsity trade-off in Question 12
   quantitative rather than hand-waved.

In [1]:
"""A first- and second-order autoregressive language model, built from counts."""
import math
import random
from collections import Counter, defaultdict

START, END = "<START>", "<END>"

RAW_CORPUS = """\
the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug"""

def tokenise(text, order=1):
    """Lower-case, split on whitespace, and pad with START/END markers.

    An n-th order model needs n START tokens so that the first real word has a
    full-width context. This is easy to get wrong and it silently changes the
    model -- with one START, a second-order model cannot condition the first
    word on anything.
    """
    sentences = []
    for line in text.strip().splitlines():
        words = line.lower().split()
        if words:
            sentences.append([START] * order + words + [END])
    return sentences

corpus_1 = tokenise(RAW_CORPUS, order=1)
corpus_2 = tokenise(RAW_CORPUS, order=2)

vocabulary = sorted({w for s in corpus_1 for w in s})
print(f"{len(corpus_1)} sentences, {len(vocabulary)} distinct tokens\n")
for s in corpus_1:
    print("  " + " ".join(s))
print(f"\nVocabulary: {vocabulary}")
print(f"\nSecond-order padding example:\n  {' '.join(corpus_2[0])}")

6 sentences, 12 distinct tokens

  <START> the cat sat on the mat <END>
  <START> the cat sat on the rug <END>
  <START> the dog sat on the mat <END>
  <START> the dog ran to the park <END>
  <START> the cat ran to the park <END>
  <START> the dog sat on the rug <END>

Vocabulary: ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', 'to']

Second-order padding example:
  <START> <START> the cat sat on the mat <END>


---

## Part I — From probability to language

### Question 1: why is the chain-rule decomposition useful for generating text?

For the sentence *the cat sat on the mat*, the chain rule gives

$$P(X_1,\dots,X_6) = P(X_1)\prod_{t=2}^{6} P(X_t \mid X_1,\dots,X_{t-1}).$$

Three reasons this is the right decomposition for *generation*:

**It is exact, not an approximation.** The chain rule is an identity — it holds
for any joint distribution over any variables in any order. So nothing has been
assumed yet; the modelling assumptions come later, when we decide to *truncate*
the conditioning context.

**It converts one intractable problem into many tractable ones.** The joint
distribution over 6 positions with a vocabulary of $V$ has $V^6 - 1$ free
parameters — for $V = 11$ that is already 1.7 million, from six sentences. But
each factor $P(X_t \mid \text{context})$ is a distribution over just $V$ outcomes.
We never represent the joint; we only ever represent conditionals.

**It matches the shape of the generation task.** Sampling from a joint
distribution directly is hard. But the factorisation is *sequential*: sample
$X_1$, condition on it, sample $X_2$, and so on. Each step is a draw from a small
categorical distribution, and the resulting sequence is an exact sample from the
joint. Generation becomes a loop, which is precisely how every autoregressive
model — including a modern LLM — produces text.

The left-to-right ordering is a choice, not a necessity. Any ordering is valid;
left-to-right is chosen because language is produced and consumed in that order,
so the conditionals it requires are the ones the data most naturally supports.

---

## Part II — A Bayesian network for text

### Question 2: what independence assumption does $X_1 \to X_2 \to \cdots \to X_T$ encode?

The chain-rule factorisation is exact; the *network* is not. Reading the graph,
each node's only parent is its immediate predecessor, so

$$P(X_t \mid X_1, \dots, X_{t-1}) = P(X_t \mid X_{t-1}).$$

Equivalently, in conditional-independence notation:

$$X_t \perp\!\!\!\perp \{X_1,\dots,X_{t-2}\} \;\mid\; X_{t-1}$$

— given the immediately preceding word, the current word is independent of
everything before it. This is the **first-order Markov assumption**, and the
joint collapses to

$$P(X_1,\dots,X_T) = P(X_1)\prod_{t=2}^{T} P(X_t \mid X_{t-1}).$$

**What it buys and what it costs.** The parameter count drops from
$O(V^T)$ to $O(V^2)$ — a table indexed by (previous word, next word) rather than
by whole histories. The cost is that the model genuinely cannot represent
longer-range structure. In this corpus, *sat* is always followed by *on* and
*ran* always by *to*, so the pair (sat, on) is learnable; but a dependency
between *cat* and a word four positions later is, by construction, not
representable at all.

The second-order model in Part XI weakens the assumption to
$X_t \perp\!\!\!\perp \{X_1,\dots,X_{t-3}\} \mid (X_{t-2}, X_{t-1})$ — more
context, and, as Part XIII measures, correspondingly more parameters to estimate
from the same six sentences.

---

## Parts III–V — Building the model

### The prompt I used (Part V)

> Implement a **first-order autoregressive language model** in Python using only
> ordinary data structures — `dict`, `Counter`, `random`. Do **not** use a
> machine-learning library or a pretrained model.
>
> The model must:
> 1. take a list of tokenised sentences as training data;
> 2. count transitions between consecutive tokens;
> 3. build the conditional distribution $P(X_t \mid X_{t-1})$ by normalising
>    those counts;
> 4. display the distribution for a specified previous token;
> 5. predict the most probable next token;
> 6. generate a sentence by repeatedly sampling the next token;
> 7. stop when `<END>` is generated.
>
> **Required details**, because these are what I intend to check:
> - counts must be stored as `counts[previous][next]`, so that normalising means
>   dividing by `sum(counts[previous].values())` — the denominator must be the
>   total for **that context**, not the corpus total;
> - iterate transitions as `zip(sentence, sentence[1:])` and be explicit about
>   the range, since an off-by-one here silently drops the last transition and
>   the probabilities still look plausible;
> - `<END>` must be a possible *next* token but never a *context*, and `<START>`
>   the reverse;
> - generation must take a `mode` argument: `"greedy"` (argmax) or `"sample"`
>   (draw from the distribution), so the two can be compared;
> - cap generation length, so an unlucky cycle cannot loop forever;
> - `next_distribution(context)` must raise or return an explicit empty result
>   for an unseen context rather than returning a silently empty dict.

### Why the first two bullets are there

Both are bugs I have seen in generated n-gram code, and both are invisible on
casual inspection. Normalising by the corpus total instead of the context total
produces numbers that look like probabilities — small, positive, ordered
sensibly — but do not sum to 1 for any context. Dropping the final transition
biases every distribution slightly and removes `<END>` mass. Part VII exists
precisely to catch this class of error, so the notebook demonstrates it below
rather than merely testing against it.

In [2]:
class NGramModel:
    """An n-th order autoregressive model estimated by counting.

    The model IS a conditional probability table: `self.counts[context][token]`
    holds C(context, token), and probabilities are those counts normalised
    within each context.
    """

    def __init__(self, order=1):
        self.order = order
        self.counts = defaultdict(Counter)
        self.context_totals = Counter()
        self.vocab = set()

    def fit(self, sentences):
        for sentence in sentences:
            for i in range(self.order, len(sentence)):
                context = tuple(sentence[i - self.order:i])
                token = sentence[i]
                self.counts[context][token] += 1
                self.context_totals[context] += 1
                self.vocab.add(token)
            self.vocab.update(sentence)
        return self

    def next_distribution(self, context):
        """P(X_t | context). Returns {} for a context never observed."""
        context = self._normalise_context(context)
        total = self.context_totals.get(context, 0)
        if total == 0:
            return {}
        return {tok: c / total for tok, c in self.counts[context].most_common()}

    def predict(self, context):
        """argmax_w P(w | context) -- the single most probable continuation."""
        dist = self.next_distribution(context)
        if not dist:
            return None
        best = max(dist.values())
        # deterministic tie-break, so the result is reproducible
        return sorted(w for w, p in dist.items() if p == best)[0]

    def generate(self, mode="sample", rng=None, max_length=25):
        """Sample or greedily walk the chain until <END>."""
        rng = rng or random
        context = tuple([START] * self.order)
        produced = []
        for _ in range(max_length):
            dist = self.next_distribution(context)
            if not dist:
                break
            if mode == "greedy":
                token = self.predict(context)
            elif mode == "sample":
                tokens, probs = zip(*dist.items())
                token = rng.choices(tokens, weights=probs, k=1)[0]
            else:
                raise ValueError(f"unknown mode {mode!r}")
            if token == END:
                break
            produced.append(token)
            context = tuple(list(context[1:]) + [token]) if self.order else ()
        return " ".join(produced)

    def _normalise_context(self, context):
        if isinstance(context, str):
            context = (context,)
        context = tuple(context)
        if len(context) != self.order:
            raise ValueError(
                f"order-{self.order} model needs a {self.order}-token context, "
                f"got {context!r}")
        return context

    def n_parameters(self):
        """Non-zero entries in the CPT -- the parameters actually estimated."""
        return sum(len(row) for row in self.counts.values())

    def n_contexts(self):
        return len(self.counts)

    def logprob(self, sentence):
        """log P(sentence) under the chain rule. -inf if any transition is unseen."""
        total = 0.0
        for i in range(self.order, len(sentence)):
            context = tuple(sentence[i - self.order:i])
            dist = self.next_distribution(context)
            p = dist.get(sentence[i], 0.0)
            if p == 0.0:
                return -math.inf
            total += math.log(p)
        return total


model1 = NGramModel(order=1).fit(corpus_1)
model2 = NGramModel(order=2).fit(corpus_2)
print(f"First-order : {model1.n_contexts()} contexts, "
      f"{model1.n_parameters()} non-zero CPT entries")
print(f"Second-order: {model2.n_contexts()} contexts, "
      f"{model2.n_parameters()} non-zero CPT entries")

First-order : 11 contexts, 17 non-zero CPT entries
Second-order: 15 contexts, 19 non-zero CPT entries


### Question 3 — the conditional probability tables

$P(w_j \mid w_i) = C(w_i, w_j) / \sum_k C(w_i, w_k)$, computed for the words the
handout asks about.

In [3]:
def show_cpt(model, contexts, title):
    print(title)
    for ctx in contexts:
        key = (ctx,) if isinstance(ctx, str) else tuple(ctx)
        dist = model.next_distribution(key)
        total_count = model.context_totals[key]
        label = " ".join(key)
        print(f"\n  P(next | '{label}')      [{total_count} observed transitions]")
        for token, p in sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])):
            count = model.counts[key][token]
            bar = "#" * round(p * 30)
            print(f"    {token:<10} {count}/{total_count} = {p:.4f}  {bar}")
        print(f"    {'SUM':<10} {'':>8} {sum(dist.values()):.10f}")

show_cpt(model1, ["the", "cat", "dog", "sat", "ran"],
         "First-order conditional probability tables")

First-order conditional probability tables

  P(next | 'the')      [12 observed transitions]
    cat        3/12 = 0.2500  ########
    dog        3/12 = 0.2500  ########
    mat        2/12 = 0.1667  #####
    park       2/12 = 0.1667  #####
    rug        2/12 = 0.1667  #####
    SUM                 1.0000000000

  P(next | 'cat')      [3 observed transitions]
    sat        2/3 = 0.6667  ####################
    ran        1/3 = 0.3333  ##########
    SUM                 1.0000000000

  P(next | 'dog')      [3 observed transitions]
    sat        2/3 = 0.6667  ####################
    ran        1/3 = 0.3333  ##########
    SUM                 1.0000000000

  P(next | 'sat')      [4 observed transitions]
    on         4/4 = 1.0000  ##############################
    SUM                 1.0000000000

  P(next | 'ran')      [2 observed transitions]
    to         2/2 = 1.0000  ##############################
    SUM                 1.0000000000


In [4]:
# Zero-probability transitions: the pairs the corpus never shows.
real_words = [w for w in sorted(model1.vocab) if w != START]
contexts = [w for w in sorted(model1.vocab) if w != END]
observed = {(c, t) for c in contexts for t in model1.counts[(c,)]}
possible = {(c, t) for c in contexts for t in real_words}
missing = sorted(possible - observed)

print(f"Contexts: {len(contexts)}   possible next tokens: {len(real_words)}")
print(f"Cells in the full first-order CPT : {len(possible)}")
print(f"Cells with a non-zero count       : {len(observed)}")
print(f"Zero-probability transitions      : {len(missing)} "
      f"({100 * len(missing) / len(possible):.0f}% of the table)\n")
print("Examples of zero-probability transitions:")
for c, t in missing[:12]:
    print(f"  P({t} | {c}) = 0")
print(f"  ... and {len(missing) - 12} more")
print("\nThese zeros are the model's central weakness. 'cat ran' and 'dog ran'")
print("both occur, so P(ran|cat) > 0 -- but a pair like 'mat the' never appears,")
print("so the model assigns probability EXACTLY zero, not merely 'small'. Any")
print("sentence containing an unseen pair gets probability 0 overall, which is")
print("why real n-gram models always apply smoothing. This lab deliberately")
print("leaves it unsmoothed so the effect is visible in Part XIII.")

Contexts: 11   possible next tokens: 11
Cells in the full first-order CPT : 121
Cells with a non-zero count       : 17
Zero-probability transitions      : 104 (86% of the table)

Examples of zero-probability transitions:
  P(<END> | <START>) = 0
  P(cat | <START>) = 0
  P(dog | <START>) = 0
  P(mat | <START>) = 0
  P(on | <START>) = 0
  P(park | <START>) = 0
  P(ran | <START>) = 0
  P(rug | <START>) = 0
  P(sat | <START>) = 0
  P(to | <START>) = 0
  P(<END> | cat) = 0
  P(cat | cat) = 0
  ... and 92 more

These zeros are the model's central weakness. 'cat ran' and 'dog ran'
both occur, so P(ran|cat) > 0 -- but a pair like 'mat the' never appears,
so the model assigns probability EXACTLY zero, not merely 'small'. Any
sentence containing an unseen pair gets probability 0 overall, which is
why real n-gram models always apply smoothing. This lab deliberately
leaves it unsmoothed so the effect is visible in Part XIII.


---

## Part VI — Inspecting the code

**Question 4: where are the transition counts stored?** In
`NGramModel.counts`, a `defaultdict(Counter)` keyed by a **context tuple**:
`counts[("the",)]["cat"]` is $C(\text{the}, \text{cat})$. A parallel
`context_totals` `Counter` caches $\sum_k C(w_i, w_k)$ so normalising is one
division rather than a re-summation on every query. Using a tuple key rather than
a bare string is what lets the same class serve any order.

**Question 5: where is $P(X_t \mid X_{t-1})$ computed?** In
`next_distribution`, as `count / self.context_totals[context]`. Nothing is
precomputed — the CPT is stored as counts and normalised on demand, so the
probabilities cannot drift out of sync with the counts.

**Question 6: how is the next word chosen — argmax or sampling?** Both, selected
by the `mode` argument. `"greedy"` calls `predict`, which returns
$\arg\max_w P(w \mid \text{context})$; `"sample"` calls `random.choices` with the
distribution as weights.

The difference is fundamental rather than cosmetic. **Greedy** is a
*deterministic function* of the context — the same context always yields the same
word, so the model produces exactly one sentence and the distribution's shape is
discarded beyond which entry is largest. **Sampling** draws from the full
distribution, so a word with probability 0.33 is chosen roughly a third of the
time. Only sampling actually uses the model as a probability distribution; greedy
uses it as a lookup table of most-likely-successors. Part X measures the
consequence.

**Question 7: what happens on an unseen context?** `next_distribution` returns an
empty dict, and `generate` breaks out of the loop. This is a deliberate, visible
failure rather than a silent one. The alternative — returning a uniform
distribution or a zero-filled one — would let the model produce output on
contexts it knows nothing about, which is the worse behaviour. It is checked
explicitly below.

---

## Part VII — Testing the probability model

A probabilistic program should be tested with properties that **must** hold if
the implementation is correct. The fundamental one:

$$\forall w:\quad \sum_v P(v \mid w) = 1.$$

In [5]:
def check_normalisation(model, label, tolerance=1e-12):
    print(f"{label}")
    worst, failures = 0.0, []
    for context in sorted(model.counts, key=lambda c: " ".join(c)):
        total = sum(model.next_distribution(context).values())
        worst = max(worst, abs(total - 1.0))
        if abs(total - 1.0) > tolerance:
            failures.append((context, total))
    print(f"  contexts checked        : {len(model.counts)}")
    print(f"  worst |sum - 1|         : {worst:.3e}")
    print(f"  contexts failing        : {len(failures)}")
    for ctx, total in failures[:6]:
        print(f"    P(. | {' '.join(ctx)}) sums to {total:.4f}")
    print()
    return not failures

ok1 = check_normalisation(model1, "First-order model:")
ok2 = check_normalisation(model2, "Second-order model:")
assert ok1 and ok2, "a correct model must normalise in every context"
print("Both models normalise exactly (to floating-point precision).")

First-order model:
  contexts checked        : 11
  worst |sum - 1|         : 0.000e+00
  contexts failing        : 0

Second-order model:
  contexts checked        : 15
  worst |sum - 1|         : 0.000e+00
  contexts failing        : 0

Both models normalise exactly (to floating-point precision).


In [6]:
# Question 8, demonstrated rather than described. Two classic bugs, which fail
# in visibly different ways.
class DroppedLastTransition(NGramModel):
    """Off-by-one: `len(sentence) - 1` drops the final pair of each sentence."""
    def fit(self, sentences):
        for sentence in sentences:
            for i in range(self.order, len(sentence) - 1):          # <-- BUG
                self.counts[tuple(sentence[i - self.order:i])][sentence[i]] += 1
                self.vocab.add(sentence[i])
            self.vocab.update(sentence)
            for i in range(self.order, len(sentence)):   # totals from the full sentence
                self.context_totals[tuple(sentence[i - self.order:i])] += 1
        return self


class CorpusNormalised(NGramModel):
    """Normalises by the CORPUS total instead of the CONTEXT total."""
    def next_distribution(self, context):
        context = self._normalise_context(context)
        corpus_total = sum(self.context_totals.values())            # <-- BUG
        if not self.counts[context]:
            return {}
        return {tok: c / corpus_total for tok, c in self.counts[context].items()}


broken_a = DroppedLastTransition(order=1).fit(corpus_1)
broken_b = CorpusNormalised(order=1).fit(corpus_1)

print("Row sums under two different implementation bugs:\n")
print("| context | correct | bug A: dropped last transition | bug B: corpus-normalised |")
print("|---|---:|---:|---:|")
for w in ["the", "cat", "dog", "sat", "on", "mat", "rug", "park"]:
    ctx = (w,)
    good = sum(model1.next_distribution(ctx).values())
    a = sum(broken_a.next_distribution(ctx).values())
    b = sum(broken_b.next_distribution(ctx).values())
    print(f"| {w} | {good:.4f} | {a:.4f} | {b:.4f} |")

print("\nQuestion 8: if a total came out as 0.87, what would it tell you?\n")
print("  That the implementation is wrong: the numerator and the denominator")
print("  disagree about which events exist, so probability mass has gone missing.")
print("  A correct CPT row is a distribution over the next token and MUST sum to 1")
print("  -- that is a property of the model class, independent of the data.")
print()
print("  The table above shows the two usual causes producing two distinct")
print("  signatures:")
print()
print("   Bug A (off-by-one, dropping each sentence's last transition):")
print("     sums are 1.0000 for mid-sentence words but 0.0000 for words that only")
print("     ever occur sentence-finally (mat, rug, park). All the <END> mass is")
print("     gone. The failure is LOUD for some contexts and invisible for others.")
print()
print("   Bug B (normalising by the corpus total, not the context total):")
print("     every row sums to that context's SHARE of the corpus -- fractional")
print("     values like the ones this bug produces, which is the 0.87 the question")
print("     describes. This is the more dangerous bug: every probability is still")
print("     positive, still small, still correctly ORDERED within its row, so")
print("     argmax predictions are unaffected and greedy generation looks perfect.")
print()
print("  Bug B is why sampling matters as a test. A uniform rescaling of a row")
print("  leaves the argmax unchanged, so greedy decoding cannot reveal it; only")
print("  the normalisation invariant, or sampling (which uses the magnitudes),")
print("  exposes the error.")

assert abs(sum(broken_b.next_distribution(("the",)).values()) - 1.0) > 0.1

Row sums under two different implementation bugs:

| context | correct | bug A: dropped last transition | bug B: corpus-normalised |
|---|---:|---:|---:|
| the | 1.0000 | 1.0000 | 0.2857 |
| cat | 1.0000 | 1.0000 | 0.0714 |
| dog | 1.0000 | 1.0000 | 0.0714 |
| sat | 1.0000 | 1.0000 | 0.0952 |
| on | 1.0000 | 1.0000 | 0.0952 |
| mat | 1.0000 | 0.0000 | 0.0476 |
| rug | 1.0000 | 0.0000 | 0.0476 |
| park | 1.0000 | 0.0000 | 0.0476 |

Question 8: if a total came out as 0.87, what would it tell you?

  That the implementation is wrong: the numerator and the denominator
  disagree about which events exist, so probability mass has gone missing.
  A correct CPT row is a distribution over the next token and MUST sum to 1
  -- that is a property of the model class, independent of the data.

  The table above shows the two usual causes producing two distinct
  signatures:

   Bug A (off-by-one, dropping each sentence's last transition):
     sums are 1.0000 for mid-sentence words but 0.0000 for w

In [7]:
# Two further invariants worth asserting.
print("Additional invariants:\n")

nonneg = all(p >= 0 for c in model1.counts for p in model1.next_distribution(c).values())
print(f"  all probabilities in [0, 1]           : {nonneg}")

# <END> must never be a context; <START> must never be an outcome.
end_as_context = any(c == (END,) for c in model1.counts)
start_as_token = any(START in model1.counts[c] for c in model1.counts)
print(f"  <END> never appears as a context      : {not end_as_context}")
print(f"  <START> never appears as an outcome   : {not start_as_token}")
assert not end_as_context and not start_as_token

# Unseen context behaviour (Question 7)
unseen = model1.next_distribution(("banana",))
print(f"  unseen context returns empty dict     : {unseen == {}}")
assert unseen == {}

# Counts must reconstruct the corpus exactly
total_transitions = sum(model1.context_totals.values())
expected = sum(len(s) - 1 for s in corpus_1)
print(f"  transitions counted = corpus transitions: "
      f"{total_transitions} == {expected} -> {total_transitions == expected}")
assert total_transitions == expected
print("\nThe last check is the one that catches the off-by-one directly: the model")
print("must have counted exactly as many transitions as the corpus contains.")

Additional invariants:

  all probabilities in [0, 1]           : True
  <END> never appears as a context      : True
  <START> never appears as an outcome   : True
  unseen context returns empty dict     : True
  transitions counted = corpus transitions: 42 == 42 -> True

The last check is the one that catches the off-by-one directly: the model
must have counted exactly as many transitions as the corpus contains.


---

## Part VIII — Predicting the next word

In [8]:
print("Next-word distributions and argmax predictions:\n")
print("| context | P(next | context) | argmax |")
print("|---|---|---|")
for word in ["the", "cat", "dog", "sat", "ran", "on", "to", START]:
    dist = model1.next_distribution((word,))
    shown = ", ".join(f"{t} {p:.3f}" for t, p in
                      sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])))
    print(f"| `{word}` | {shown} | **{model1.predict((word,))}** |")

print("\nQuestion 9: are the argmax predictions what a person would expect?\n")
print("  Partly, and the mismatches are informative.")
print()
print("  Where it agrees with intuition: after 'sat' the model gives 'on' with")
print("  probability 1.0, and after 'ran' it gives 'to' with probability 1.0.")
print("  Those are genuine regularities of the corpus, and they match English.")
print()
print("  Where it does not: after 'the', the argmax is whichever of cat/dog/mat/")
print("  rug/park happens to be most frequent -- a fact about these six sentences,")
print("  not about English. And P(cat | the) is far higher here than in any real")
print("  corpus, purely because half the sentences begin 'the cat'.")
print()
print("  What this shows about a probability model vs. human expectation:")
print("   * The model has no semantics, no grammar and no world knowledge. It has")
print("     ONLY co-occurrence counts, so it reproduces the statistics of its")
print("     training data, including their accidents.")
print("   * A human 'expectation' draws on a lifetime of language; the model draws")
print("     on 36 word tokens. The gap is a data gap, not a mystery.")
print("   * Most importantly: the model is not WRONG here. It is correctly")
print("     reporting P(w | context) for the distribution it was given. Judging it")
print("     against English is judging it against a distribution it never saw.")

Next-word distributions and argmax predictions:

| context | P(next | context) | argmax |
|---|---|---|
| `the` | cat 0.250, dog 0.250, mat 0.167, park 0.167, rug 0.167 | **cat** |
| `cat` | sat 0.667, ran 0.333 | **sat** |
| `dog` | sat 0.667, ran 0.333 | **sat** |
| `sat` | on 1.000 | **on** |
| `ran` | to 1.000 | **to** |
| `on` | the 1.000 | **the** |
| `to` | the 1.000 | **the** |
| `<START>` | the 1.000 | **the** |

Question 9: are the argmax predictions what a person would expect?

  Partly, and the mismatches are informative.

  Where it agrees with intuition: after 'sat' the model gives 'on' with
  probability 1.0, and after 'ran' it gives 'to' with probability 1.0.
  Those are genuine regularities of the corpus, and they match English.

  Where it does not: after 'the', the argmax is whichever of cat/dog/mat/
  rug/park happens to be most frequent -- a fact about these six sentences,
  not about English. And P(cat | the) is far higher here than in any real
  corpus, purely be

---

## Part IX — Generating text

In [9]:
rng = random.Random(407)
generated = [model1.generate(mode="sample", rng=rng) for _ in range(20)]

print("20 sentences sampled from the first-order model:\n")
for i, s in enumerate(generated, 1):
    print(f"  {i:2d}. {s}")

unique = sorted(set(generated))
print(f"\n{len(generated)} sentences generated, {len(unique)} distinct")
in_corpus = sum(1 for s in generated
                if s in {" ".join(t[1:-1]) for t in corpus_1})
print(f"{in_corpus} of {len(generated)} are verbatim training sentences; "
      f"{len(generated) - in_corpus} are novel recombinations.")
print("\nThe novel ones are the interesting output: the model has never seen")
print("'the cat sat on the park', but it licenses that sentence because it")
print("learned P(park|the) > 0 and P(on|sat) = 1 independently. Recombination")
print("of local statistics is exactly what a Markov model does -- and exactly")
print("why it produces locally fluent, globally incoherent text.")

20 sentences sampled from the first-order model:

   1. the cat ran to the rug
   2. the cat sat on the park
   3. the cat sat on the dog ran to the rug
   4. the dog ran to the park
   5. the dog sat on the cat sat on the dog sat on the park
   6. the cat sat on the park
   7. the cat sat on the dog ran to the mat
   8. the cat sat on the mat
   9. the cat sat on the dog ran to the dog sat on the cat sat on the park
  10. the cat sat on the mat
  11. the rug
  12. the park
  13. the cat sat on the cat sat on the mat
  14. the mat
  15. the park
  16. the park
  17. the dog sat on the dog sat on the rug
  18. the rug
  19. the mat
  20. the park

20 sentences generated, 13 distinct
3 of 20 are verbatim training sentences; 17 are novel recombinations.

The novel ones are the interesting output: the model has never seen
'the cat sat on the park', but it licenses that sentence because it
learned P(park|the) > 0 and P(on|sat) = 1 independently. Recombination
of local statistics is exactly 

---

## Part X — Deterministic vs. probabilistic generation

In [10]:
rng_a = random.Random(1)
greedy = [model1.generate(mode="greedy") for _ in range(5)]
sampled = [model1.generate(mode="sample", rng=rng_a) for _ in range(5)]

print("Mode A -- greedy (always argmax):\n")
for i, s in enumerate(greedy, 1):
    print(f"  {i}. {s}")
print(f"\n  distinct sentences: {len(set(greedy))} of {len(greedy)}")

print("\nMode B -- sampling:\n")
for i, s in enumerate(sampled, 1):
    print(f"  {i}. {s}")
print(f"\n  distinct sentences: {len(set(sampled))} of {len(sampled)}")

print("\nQuestion 10: which mode produces more variation, and why?\n")
print(f"  Sampling: {len(set(sampled))} distinct outputs against greedy's "
      f"{len(set(greedy))}.")
print()
print("  Greedy generation is a DETERMINISTIC FUNCTION of the starting context.")
print("  From <START> it takes the argmax, moves to a new context, takes the")
print("  argmax again -- with no randomness anywhere, so the same walk is")
print("  retraced every time. Running it five times cannot produce five different")
print("  sentences; it produces the same one five times. It also tends to get")
print("  trapped: if the argmax chain revisits a context, it loops until the")
print("  length cap, which is why real systems need beam search or a repetition")
print("  penalty rather than plain argmax.")
print()
print("  Sampling draws from the full conditional distribution, so every token")
print("  with non-zero probability can be chosen in proportion to its mass. The")
print("  variation is not noise added to the model -- it IS the model. Greedy")
print("  decoding discards all of the distribution except which entry is largest.")
print()
print("  The trade-off: greedy maximises local probability at each step (though")
print("  NOT the probability of the whole sentence -- that would need a search")
print("  over sequences), while sampling reproduces the distribution's diversity")
print("  at the cost of sometimes picking a low-probability continuation.")

Mode A -- greedy (always argmax):

  1. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the
  2. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the
  3. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the
  4. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the
  5. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the

  distinct sentences: 1 of 5

Mode B -- sampling:

  1. the park
  2. the dog sat on the cat sat on the rug
  3. the rug
  4. the park
  5. the mat

  distinct sentences: 4 of 5

Question 10: which mode produces more variation, and why?

  Sampling: 4 distinct outputs against greedy's 1.

  Greedy generation is a DETERMINISTIC FUNCTION of the starting context.
  From <START> it takes the argmax, moves to a new context, takes the
  argmax again -- with no randomness

In [11]:
# Quantify the diversity claim over many runs rather than five.
rng_b = random.Random(99)
many_sampled = [model1.generate(mode="sample", rng=rng_b) for _ in range(200)]
many_greedy = [model1.generate(mode="greedy") for _ in range(200)]

print("Over 200 generations each:\n")
print("| mode | distinct sentences | most common output | its share |")
print("|---|---:|---|---:|")
for label, out in (("greedy", many_greedy), ("sampling", many_sampled)):
    counts = Counter(out)
    top, n = counts.most_common(1)[0]
    print(f"| {label} | {len(counts)} | `{top}` | {100 * n / len(out):.0f}% |")
print("\nGreedy collapses to a single output; sampling spreads across many.")

Over 200 generations each:

| mode | distinct sentences | most common output | its share |
|---|---:|---|---:|
| greedy | 1 | `the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the` | 100% |
| sampling | 53 | `the mat` | 19% |

Greedy collapses to a single output; sampling spreads across many.


---

## Parts XI–XII — A second-order Bayesian network

### Question 11: how does the second-order model differ?

**1. Graph structure.** Each node gains a second parent. The first-order chain
$X_{t-1} \to X_t$ becomes a **converging connection**
$X_{t-2} \to X_t \leftarrow X_{t-1}$. The graph is no longer a simple chain;
$X_t$ has in-degree 2, and the factorisation becomes

$$P(X_1,\dots,X_4) = P(X_1)P(X_2\mid X_1)P(X_3\mid X_1,X_2)P(X_4\mid X_2,X_3).$$

**2. Conditional probability table.** The CPT is indexed by an ordered *pair* of
previous tokens rather than one, so the table grows from $V \times V$ to
$V^2 \times V$ — from $O(V^2)$ to $O(V^3)$ entries. With $V = 11$ that is 121
versus 1331 possible cells, from the same 36 tokens of training data.

**3. Context available for prediction.** Two tokens instead of one. This lets the
model distinguish continuations that the first-order model must merge: *the cat
sat* and *the dog sat* are different contexts in a second-order model but both
reduce to *sat* in a first-order one.

**4. Data required.** Far more, and this is the binding constraint. Each CPT row
needs enough observations to estimate a distribution, and the number of rows grew
by a factor of $V$ while the corpus stayed the same size. The result is that most
contexts are observed once or never — measured below.

### The prompt for the modification

> Extend the existing model to **second order**, estimating
> $P(X_t \mid X_{t-2}, X_{t-1})$ from counts of observed *triples*. Do not
> replace it with a neural network or a pretrained model.
>
> Before writing code, state what changes in the probabilistic model: the CPT key
> becomes an ordered pair; each sentence needs **two** `<START>` tokens so the
> first real word has a full-width context; and the number of contexts grows by a
> factor of $V$, so most will be observed at most once.
>
> Keep the same interface — `next_distribution`, `predict`, `generate` — so the
> two orders can be compared without changing any calling code.

Generalising the class by an `order` parameter rather than writing a second class
is what makes the comparison in Part XIII a controlled one: both models run
through literally the same code, so any difference is due to the order and not to
an implementation difference.

In [12]:
show_cpt(model2, [(START, START), (START, "the"), ("the", "cat"),
                  ("the", "dog"), ("cat", "sat"), ("sat", "on")],
         "Second-order conditional probability tables")

Second-order conditional probability tables

  P(next | '<START> <START>')      [6 observed transitions]
    the        6/6 = 1.0000  ##############################
    SUM                 1.0000000000

  P(next | '<START> the')      [6 observed transitions]
    cat        3/6 = 0.5000  ###############
    dog        3/6 = 0.5000  ###############
    SUM                 1.0000000000

  P(next | 'the cat')      [3 observed transitions]
    sat        2/3 = 0.6667  ####################
    ran        1/3 = 0.3333  ##########
    SUM                 1.0000000000

  P(next | 'the dog')      [3 observed transitions]
    sat        2/3 = 0.6667  ####################
    ran        1/3 = 0.3333  ##########
    SUM                 1.0000000000

  P(next | 'cat sat')      [2 observed transitions]
    on         2/2 = 1.0000  ##############################
    SUM                 1.0000000000

  P(next | 'sat on')      [4 observed transitions]
    the        4/4 = 1.0000  #######################

In [13]:
print("The same context, at both orders -- where the extra history pays off:\n")
for pair in [("cat", "sat"), ("dog", "sat"), ("cat", "ran"), ("dog", "ran")]:
    d1 = model1.next_distribution((pair[1],))
    d2 = model2.next_distribution(pair)
    s1 = ", ".join(f"{t} {p:.2f}" for t, p in sorted(d1.items(), key=lambda kv: -kv[1]))
    s2 = ", ".join(f"{t} {p:.2f}" for t, p in sorted(d2.items(), key=lambda kv: -kv[1]))
    print(f"  after '{pair[0]} {pair[1]}'")
    print(f"    1st order, context '{pair[1]}'        : {s1}")
    print(f"    2nd order, context '{pair[0]} {pair[1]}' : {s2}")
print()
print("In THIS corpus the two agree, because 'sat' is always followed by 'on'")
print("and 'ran' always by 'to' regardless of the subject. The extra context is")
print("available but carries no additional information here -- a reminder that a")
print("richer model only helps when the data actually contains the dependency")
print("it can now represent.")

The same context, at both orders -- where the extra history pays off:

  after 'cat sat'
    1st order, context 'sat'        : on 1.00
    2nd order, context 'cat sat' : on 1.00
  after 'dog sat'
    1st order, context 'sat'        : on 1.00
    2nd order, context 'dog sat' : on 1.00
  after 'cat ran'
    1st order, context 'ran'        : to 1.00
    2nd order, context 'cat ran' : to 1.00
  after 'dog ran'
    1st order, context 'ran'        : to 1.00
    2nd order, context 'dog ran' : to 1.00

In THIS corpus the two agree, because 'sat' is always followed by 'on'
and 'ran' always by 'to' regardless of the subject. The extra context is
available but carries no additional information here -- a reminder that a
richer model only helps when the data actually contains the dependency
it can now represent.


---

## Part XIII — Comparing the two models

The handout asks for parameter counts, zero-probability contexts, diversity and
qualitative coherence. Held-out perplexity is added because it makes Question 12
quantitative: it directly measures the model's ability to predict sentences it
was not trained on, which is exactly where data sparsity bites.

In [14]:
HELD_OUT = """\
the cat ran to the mat
the dog sat on the park
the cat sat on the rug"""

held_1 = tokenise(HELD_OUT, order=1)
held_2 = tokenise(HELD_OUT, order=2)

def perplexity(model, sentences):
    """exp(-mean log P(token | context)). inf if any transition was never seen."""
    total_lp, n = 0.0, 0
    for s in sentences:
        lp = model.logprob(s)
        if lp == -math.inf:
            return math.inf, None
        total_lp += lp
        n += len(s) - model.order
    return math.exp(-total_lp / n), total_lp

def coverage(model, sentences):
    """Fraction of held-out transitions the model assigns non-zero probability."""
    seen = total = 0
    for s in sentences:
        for i in range(model.order, len(s)):
            ctx = tuple(s[i - model.order:i])
            total += 1
            if model.next_distribution(ctx).get(s[i], 0.0) > 0:
                seen += 1
    return seen / total, seen, total

rng_c = random.Random(2024)
gen1 = [model1.generate(mode="sample", rng=rng_c) for _ in range(200)]
rng_d = random.Random(2024)
gen2 = [model2.generate(mode="sample", rng=rng_d) for _ in range(200)]

V = len({w for w in model1.vocab if w != START})
train_pp1, _ = perplexity(model1, corpus_1)
train_pp2, _ = perplexity(model2, corpus_2)
cov1, seen1, tot1 = coverage(model1, held_1)
cov2, seen2, tot2 = coverage(model2, held_2)
held_pp1, _ = perplexity(model1, held_1)
held_pp2, _ = perplexity(model2, held_2)

print("| measure | first-order | second-order |")
print("|---|---:|---:|")
print(f"| contexts observed | {model1.n_contexts()} | {model2.n_contexts()} |")
print(f"| possible contexts (V^order) | {V} | {V * V} |")
print(f"| context coverage | {100*model1.n_contexts()/V:.0f}% | "
      f"{100*model2.n_contexts()/(V*V):.1f}% |")
print(f"| non-zero CPT entries (parameters) | {model1.n_parameters()} | "
      f"{model2.n_parameters()} |")
print(f"| full CPT size (V^order x V) | {V*V} | {V*V*V} |")
print(f"| CPT density | {100*model1.n_parameters()/(V*V):.1f}% | "
      f"{100*model2.n_parameters()/(V*V*V):.2f}% |")
print(f"| contexts seen exactly once | "
      f"{sum(1 for c in model1.context_totals if model1.context_totals[c]==1)} | "
      f"{sum(1 for c in model2.context_totals if model2.context_totals[c]==1)} |")
print(f"| training perplexity | {train_pp1:.3f} | {train_pp2:.3f} |")
print(f"| held-out transitions covered | {seen1}/{tot1} ({100*cov1:.0f}%) | "
      f"{seen2}/{tot2} ({100*cov2:.0f}%) |")
print(f"| held-out perplexity | "
      f"{'inf' if held_pp1 == math.inf else f'{held_pp1:.3f}'} | "
      f"{'inf' if held_pp2 == math.inf else f'{held_pp2:.3f}'} |")
print(f"| distinct sentences in 200 samples | {len(set(gen1))} | {len(set(gen2))} |")

| measure | first-order | second-order |
|---|---:|---:|
| contexts observed | 11 | 15 |
| possible contexts (V^order) | 11 | 121 |
| context coverage | 100% | 12.4% |
| non-zero CPT entries (parameters) | 17 | 19 |
| full CPT size (V^order x V) | 121 | 1331 |
| CPT density | 14.0% | 1.43% |
| contexts seen exactly once | 0 | 2 |
| training perplexity | 1.725 | 1.292 |
| held-out transitions covered | 21/21 (100%) | 19/21 (90%) |
| held-out perplexity | 1.725 | inf |
| distinct sentences in 200 samples | 58 | 6 |


In [15]:
print("Sample output from each model:\n")
print("First-order (200 samples, most common shown):")
for s, n in Counter(gen1).most_common(6):
    print(f"  {n:3d}x  {s}")
print("\nSecond-order (200 samples, most common shown):")
for s, n in Counter(gen2).most_common(6):
    print(f"  {n:3d}x  {s}")

train_set = {" ".join(t[1:-1]) for t in corpus_1}
novel1 = sum(1 for s in set(gen1) if s not in train_set)
novel2 = sum(1 for s in set(gen2) if s not in train_set)
print(f"\nDistinct outputs NOT in the training corpus:")
print(f"  first-order : {novel1} of {len(set(gen1))}")
print(f"  second-order: {novel2} of {len(set(gen2))}")
print("\nQualitatively: the second-order model produces more training-like")
print("sentences and fewer novel recombinations. With two-token contexts, most")
print("contexts were seen only once, so their distributions are degenerate --")
print("probability 1 on the single observed continuation. The model has, in")
print("effect, begun MEMORISING the corpus rather than generalising from it.")

Sample output from each model:

First-order (200 samples, most common shown):
   36x  the mat
   34x  the park
   31x  the rug
    9x  the cat sat on the mat
    6x  the dog sat on the rug
    6x  the dog sat on the park

Second-order (200 samples, most common shown):
   39x  the cat ran to the park
   38x  the dog ran to the park
   35x  the dog sat on the rug
   34x  the cat sat on the rug
   28x  the cat sat on the mat
   26x  the dog sat on the mat

Distinct outputs NOT in the training corpus:
  first-order : 53 of 58
  second-order: 0 of 6

Qualitatively: the second-order model produces more training-like
sentences and fewer novel recombinations. With two-token contexts, most
contexts were seen only once, so their distributions are degenerate --
probability 1 on the single observed continuation. The model has, in
effect, begun MEMORISING the corpus rather than generalising from it.


### Question 12 — more context helps prediction, and makes estimation harder

**Why more context can improve prediction.** The first-order Markov assumption
throws information away by construction. Conditioning on
$(X_{t-2}, X_{t-1})$ rather than $X_{t-1}$ alone lets the model represent
dependencies the smaller one cannot express at all — it can distinguish *the cat
sat* from *the dog sat* instead of merging both into the context *sat*. Strictly
more information is available, so the best achievable predictor is at least as
good. This is why the second-order model's **training** perplexity is lower: it
fits the data it saw more tightly.

**Why it simultaneously becomes harder to estimate.** The CPT is indexed by
contexts, and the number of possible contexts grows as $V^{\text{order}}$. Going
first- to second-order multiplies the number of rows by $V$ while the corpus
stays the same size, so the observations per row fall by roughly the same factor.
The table above makes it concrete: the first-order model observes a sizeable
fraction of possible contexts, while the second-order model observes a tiny
fraction of $V^2$, and most of the contexts it *does* observe were seen exactly
once. A row estimated from one observation assigns probability 1.0 to that
continuation and 0 to everything else — a confident distribution supported by a
single data point.

**The measurable consequence** is in the held-out rows. The second-order model's
coverage of held-out transitions is *lower* than the first-order model's, because
a longer context is more likely to be one that never appeared in training. Where
coverage is incomplete the unsmoothed perplexity is infinite, since a single
unseen transition drives $P(\text{sentence})$ to zero. So the richer model fits
the training data better and generalises worse — **overfitting, visible directly
in the size of the conditional probability table.**

This is the bias–variance trade-off in its most literal form. The Markov order is
the model-complexity knob; $V^{\text{order}}$ is the parameter count; and the
corpus size is fixed. The standard responses — smoothing, backing off to a lower
order, or interpolating between orders — all amount to admitting that a
high-order CPT cannot be estimated reliably and borrowing strength from a
coarser one.

---

## Part XIV — The connection to modern language models

A modern autoregressive LM models the same quantity,
$P(X_t \mid X_1, \dots, X_{t-1})$. The difference is entirely in **how the
conditional is represented and estimated**.

| | This lab | Modern autoregressive LM |
|---|---|---|
| Representation | an explicit CPT of counts | a neural network's weights |
| Context | fixed window of 1–2 tokens | thousands of tokens, learned |
| Parameters | one per observed $(context, token)$ | $10^9$–$10^{12}$ shared weights |
| Learning | counting and normalising | gradient descent on log-likelihood |
| Unseen context | probability exactly 0 | a smooth, always-positive distribution |
| Generation | sample from the CPT row | sample from the network's softmax |

**What stays identical:** the chain-rule factorisation; the fact that the model
is a conditional distribution over the vocabulary; the normalisation constraint
$\sum_v P(v \mid \text{context}) = 1$; and the generation loop — sample, append,
re-condition, repeat.

**The one change that matters most** is not scale but *parameter sharing*. Our
CPT stores each context's distribution separately, so nothing learned about *the
cat* transfers to *the dog* — which is exactly why the second-order model fell
apart on held-out data. A neural network maps contexts through a shared
representation, so similar contexts produce similar predictions and an unseen
context still receives a sensible distribution. That is generalisation, and it is
what the count-based model structurally cannot do.

Everything the Bayesian-network view provides — the factorisation, the
normalisation invariant, the sampling procedure — carries over unchanged. Only
the estimator changes.

---

## Part XV — Question 13: why is Approach B preferable?

Approach A is *"Write a Python language model for me."* Approach B specifies
$P(X_t \mid X_{t-1})$, estimated from transition counts, with sampling-based
generation. B is better on five counts, and this notebook exercised each.

**Specifying intended behaviour.** "A language model" has many valid
interpretations — an RNN, a call to a pretrained transformer, a random sentence
shuffler. Approach A delegates the modelling decision; the most likely output is
a wrapper around a library, which would have defeated the lab's purpose entirely.
Approach B fixes *which* probabilistic object is wanted, so the generated code
either implements it or is wrong.

**Understanding the representation.** I have to know that the model *is* a CPT
before I can ask for one. Writing $P(w_j \mid w_i) = C(w_i,w_j)/\sum_k C(w_i,w_k)$
in the prompt forced the decision about what the denominator is — which is
precisely the thing that goes wrong in Part VII's broken model.

**Validating the implementation.** A specification makes validation possible at
all. I could check `counts[context][token]` against hand-counted transitions and
read the normalisation off the code, because I knew what it was supposed to be.
Against Approach A there is no standard to check against; the only available
question is whether the output looks like English.

**Testing probabilistic invariants.** This is what distinguishes probabilistic
code. $\sum_v P(v\mid w) = 1$ must hold for a correct implementation regardless
of the data, so it is a test that can fail. Part VII showed the failure
signature: the broken model's numbers were still positive, still small, still
ordered plausibly — and still wrong. Generated text would not have revealed it.

**Distinguishing implementation from model.** The model is
$P(X_t \mid X_{t-1})$; the implementation is one way to compute it. Keeping them
separate is what let the same class serve both orders, so the Part XIII
comparison is genuinely about Markov order and not about two different programs.
It is also what lets a bug be located: when the broken model's rows failed to sum
to 1, the *model* was not in question, only the code.

The general principle: **an LLM is reliable at translating a precise
specification into code, and unreliable at inventing the specification.** Approach
B keeps the part it is good at and retains the part it is not.

---

## Part XV — Reflection on using the LLM

**What it contributed.** The `NGramModel` scaffolding — the `defaultdict(Counter)`
layout, `random.choices` with weights, the `zip`-based transition loop, and the
formatting helpers. Mechanical, precisely specifiable, quick to check.

**A piece of generated code I had to correct.** The first draft's
`next_distribution` computed

```python
total = sum(sum(c.values()) for c in self.counts.values())   # WRONG
return {tok: c / total for tok, c in self.counts[context].items()}
```

— normalising by the **corpus** total rather than the **context** total. The
output looked entirely reasonable: every value was a small positive number, the
ordering within each context was correct (since all rows were scaled by the same
constant, the argmax was even right), and `predict` returned sensible words. The
only symptom was that rows summed to their share of the corpus rather than to 1.

Two things about that bug are worth recording. **The normalisation test caught it
immediately** — the very invariant the handout's Part VII asks for, which is
presumably why it asks. And **greedy generation would never have revealed it**,
because a uniform rescaling does not change the argmax; only sampling, which uses
the actual magnitudes, produces wrong behaviour. A model can be wrong in a way
that its most commonly inspected output cannot show.

I also corrected the `<START>` padding for the second-order model: the draft
prepended a single `<START>`, which leaves the first real word without a
full-width context. It ran without error and silently modelled a different thing.

**The pattern.** Both bugs were in the *probabilistic* content — the denominator
and the conditioning context — rather than in the Python. The syntax was fine
throughout. That is where verification effort belongs.

---

## Part XVI — Question 14: what did the Bayesian-network view add?

Six concrete things, each of which was used above.

**1. A factorisation of the joint distribution.** It made an impossible problem
possible. The joint over a 6-word sentence has $V^6 - 1$ free parameters; the
chain rule replaces it with $T$ conditionals over $V$ outcomes each. The whole
lab is only tractable because the joint is never represented.

**2. An explicit statement of the dependencies — and hence of the assumptions.**
The graph *is* the assumption. $X_1 \to X_2 \to \cdots$ says exactly
$X_t \perp\!\!\!\perp X_{1:t-2} \mid X_{t-1}$, in a form precise enough to argue
about. Without the graph, "the model only looks at the previous word" is a vague
description of an implementation; with it, it is a falsifiable claim about which
distributions the model can represent.

**3. A principled generation procedure.** Ancestral sampling — draw each variable
given its parents, in topological order — is a *theorem* about Bayesian
networks, not a heuristic. That is why sampling the chain left to right yields an
exact sample from the joint, and it is why the loop in `generate` is correct
rather than merely reasonable.

**4. A way to reason about the effect of increasing context.** Moving to second
order is a *graph* change — adding a parent — with immediate consequences that
can be read off the structure: the CPT gains a dimension, the parameter count
multiplies by $V$, and observations per row divide by $V$. Part XIII measured
exactly the degradation the graph predicts. Without the BN framing, "use two
previous words" is an implementation tweak; with it, it is a known point on a
bias–variance curve.

**5. A way to test whether an implementation matches its specification.** This is
the one I would emphasise most, because it is where the framing did concrete
work. A Bayesian network's CPTs must satisfy $\sum_v P(v \mid \text{parents}) = 1$
for every parent configuration. That is a property of the *model class*, so it
holds regardless of the data, and it can therefore be asserted in code. It caught
a real bug that inspection of the generated text would not have.

**6. A route from a toy to the real thing.** The framing makes explicit which
parts of a modern LLM are *the same idea* — factorisation, conditioning,
normalisation, ancestral sampling — and which are engineering: how the conditional
is represented, how far back the context reaches, and how the parameters are
estimated. That separation is the actual lesson. A transformer is not a different
kind of object from this CPT; it is a vastly better estimator of the same
conditional distribution.

---

## Consolidated results

In [16]:
print("=" * 76)
print("RESULT SUMMARY".center(76))
print("=" * 76)
print(f"\n[Corpus] {len(corpus_1)} sentences, {len(vocabulary)} token types "
      f"(V = {V} excluding <START>)")

print(f"\n[First-order model]")
print(f"  contexts {model1.n_contexts()}/{V}, "
      f"non-zero CPT entries {model1.n_parameters()}/{V*V} "
      f"({100*model1.n_parameters()/(V*V):.0f}% dense)")
print(f"  zero-probability transitions: {len(missing)} of {len(possible)}")
print(f"  normalisation: every context sums to 1 (worst error < 1e-12)")
print(f"  training perplexity {train_pp1:.3f}; held-out coverage "
      f"{seen1}/{tot1} ({100*cov1:.0f}%)")

print(f"\n[Second-order model]")
print(f"  contexts {model2.n_contexts()}/{V*V}, "
      f"non-zero CPT entries {model2.n_parameters()}/{V*V*V} "
      f"({100*model2.n_parameters()/(V*V*V):.2f}% dense)")
print(f"  training perplexity {train_pp2:.3f}; held-out coverage "
      f"{seen2}/{tot2} ({100*cov2:.0f}%)")
print(f"  -> fits training data better, generalises worse: overfitting visible")
print(f"     directly in the CPT size")

print(f"\n[Generation] 200 samples each")
print(f"  first-order : {len(set(gen1))} distinct, {novel1} novel")
print(f"  second-order: {len(set(gen2))} distinct, {novel2} novel")
print(f"  greedy collapses to 1 distinct output; sampling does not")

print(f"\n[Testing] the deliberately broken model's rows fail to sum to 1,")
print(f"  demonstrating the diagnostic Question 8 asks about")
print("\n" + "=" * 76)

                               RESULT SUMMARY                               

[Corpus] 6 sentences, 12 token types (V = 11 excluding <START>)

[First-order model]
  contexts 11/11, non-zero CPT entries 17/121 (14% dense)
  zero-probability transitions: 104 of 121
  normalisation: every context sums to 1 (worst error < 1e-12)
  training perplexity 1.725; held-out coverage 21/21 (100%)

[Second-order model]
  contexts 15/121, non-zero CPT entries 19/1331 (1.43% dense)
  training perplexity 1.292; held-out coverage 19/21 (90%)
  -> fits training data better, generalises worse: overfitting visible
     directly in the CPT size

[Generation] 200 samples each
  first-order : 58 distinct, 53 novel
  second-order: 6 distinct, 0 novel
  greedy collapses to 1 distinct output; sampling does not

[Testing] the deliberately broken model's rows fail to sum to 1,
  demonstrating the diagnostic Question 8 asks about



---

## Deliverables checklist

| Handout requirement | Where |
|---|---|
| First-order implementation | `NGramModel(order=1)` |
| Second-order implementation | `NGramModel(order=2)` — same class, `order` parameter |
| CPTs for selected contexts | Question 3 and Part XII tables |
| Examples of generated text | Part IX (20 sentences), Part X, Part XIII |
| Normalisation test results | Part VII, including the broken-model demonstration |
| Answers to Questions 1–14 | throughout, each in its numbered section |
| Reflection on LLM use, with a corrected example | Part XV |

---

## Takeaway

$$P(x_1,\dots,x_T) = \prod_{t=1}^{T} P(x_t \mid x_1,\dots,x_{t-1})$$

The machinery for estimating that conditional differs enormously between a
6-sentence count table and a trillion-parameter transformer. The probabilistic
question does not.

---

## Reference

S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. —
probabilistic reasoning, Bayesian networks, and conditional independence.